# Fine-Tuning Mistral 7B on 1000+ Conversations (Unsloth + Free Colab T4)

### Instructions:
1. In Google Colab, go to **Runtime > Change runtime type** and ensure **T4 GPU** is selected.
2. Upload your 1000+ conversation dataset (`.json` or `.jsonl`) directly to Colab or place it in your Google Drive.
3. Run each cell sequentially from top to bottom.

In [ ]:
# 1. Install Unsloth and training dependencies
!pip install -q -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q -U trl peft accelerate transformers datasets bitsandbytes

import os
import torch
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset
from google.colab import drive

print("Environment initialized!")

In [ ]:
# 2. Mount Google Drive
# Mounting early ensures checkpoints and exported models are saved directly to your Drive!
drive.mount('/content/drive')
print("Google Drive connected!")

In [ ]:
# 3. Load Mistral 7B Instruct v0.3 in 4-bit
max_seq_length = 2048 # Adjust to 4096 if your conversations are very long
dtype = None # Auto-detects float16 on T4 GPU
load_in_4bit = True # Uses only ~6GB of the 15GB T4 VRAM

print("Loading Mistral 7B Instruct v0.3...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Add LoRA Adapters (Rank 16 provides capacity for 1000+ dialogues without overfitting)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("Model and LoRA configured successfully!")

In [ ]:
# 4. Load & Format 1000+ Conversations Dataset
# Enter your dataset filename here (upload it to Colab files on the left sidebar or Google Drive):
dataset_filename = "conversations_1000.jsonl"  # Change this to your file name!

if not os.path.exists(dataset_filename):
    drive_check = os.path.join("/content/drive/MyDrive", dataset_filename)
    if os.path.exists(drive_check):
        dataset_filename = drive_check
    else:
        raise FileNotFoundError(f"Please upload '{dataset_filename}' to Colab or place it in Google Drive!")

print(f"Loading dataset from {dataset_filename}...")
raw_dataset = load_dataset("json", data_files=dataset_filename, split="train")
print(f"Loaded {len(raw_dataset)} conversation records!")

# Universal Chat Formatter (supports messages list, ShareGPT, or instruction/output)
def format_prompts(batch):
    formatted = []
    if "messages" in batch:
        for msgs in batch["messages"]:
            formatted.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False))
    elif "conversations" in batch:
        for conv in batch["conversations"]:
            msgs = []
            for turn in conv:
                role = "user" if turn.get("from") in ["human", "user"] else "assistant"
                msgs.append({"role": role, "content": turn.get("value", "")})
            formatted.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False))
    elif "instruction" in batch and "output" in batch:
        for inst, inp, out in zip(batch["instruction"], batch.get("input", [""]*len(batch["instruction"])), batch["output"]):
            content = f"{inst}\n{inp}".strip() if inp else inst
            msgs = [{"role": "user", "content": content}, {"role": "assistant", "content": out}]
            formatted.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False))
    return {"text": formatted}

dataset = raw_dataset.map(format_prompts, batched=True)
print("Chat template applied successfully! Sample prompt preview:")
print(dataset[0]["text"][:300] + "...")

In [ ]:
# 5. Configure Trainer Optimized for 1000+ Conversations
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # Effective batch size = 8
        warmup_ratio = 0.05,
        num_train_epochs = 2, # 2 full epochs is optimal for 1000+ conversations (~250 steps, ~20-25 mins)
        learning_rate = 1.5e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "/content/drive/MyDrive/lisa_checkpoints",
        save_strategy = "steps",
        save_steps = 100, # Saves checkpoints every 100 steps directly to Google Drive
    ),
)
print("Trainer initialized!")

In [ ]:
# 6. Train the Model
print("Starting training on 1000+ conversations...")
trainer_stats = trainer.train()
print("Training complete! Model converged.")

In [ ]:
# 7. Export Model to GGUF (Q4_K_M) and Save to Google Drive
output_name = "lisa-mistral-7b-v2"
print(f"Exporting {output_name} to GGUF (q4_k_m)...")
model.save_pretrained_gguf(output_name, tokenizer, quantization_method = "q4_k_m")

# Copy directly to Google Drive
src_file = f"{output_name}-q4_k_m.gguf"
dest_file = f"/content/drive/MyDrive/{output_name}-q4km.gguf"
if os.path.exists(src_file):
    print(f"Copying {src_file} to Google Drive...")
    !cp {src_file} {dest_file}
    print(f"SUCCESS! Your fine-tuned GGUF is in your Google Drive: {dest_file}")
    print("You can now download it directly to your PC!")
else:
    print("Exported file:", src_file)